# Papermill driver for preprocessing (new CAP paths)

Runs `001_Preprocessing.ipynb` once for each included sample in the frozen cohort release.
Outputs are stored under the configured `CAP_DATA_ROOT/runs/<CAP_RUN_ID>/01_preprocessing`
stage. This driver does not create a new run ID and does not skip samples just because an
executed notebook already exists.

- Set `RUN_DEMO = True` in the settings cell to rerun only the two demo samples.
- A full run resets the `01_Preprocessing` stage before starting. A demo run only replaces
  the selected samples' outputs and writes a separate demo QC summary.
- Failed samples retain their partial notebook and log; the combined QC summary is written
  only when every selected sample succeeds.

**Destructive behavior:** a full run deletes the existing `01_Preprocessing` stage for the
current `CAP_RUN_ID` before rerunning it. Other stages and the `00_cohort` stage are left alone.


## Run settings

In [1]:
RUN_DEMO = True
DEMO_IDS = [
    "GSM4909292_HER2-MH0069",
    "GSM4909272_N-MH0021-Total",
]
COHORT_RELEASE_NAME = "cohort_release_2026-11-13.csv"

# None uses the kernelspec recorded in 001_Preprocessing.ipynb.
# Set this to the registered kernelspec name only if it differs.
KERNEL_NAME = None


## Resolve paths and validate inputs

In [2]:
from pathlib import Path
import shutil

import pandas as pd
import papermill as pm

import json
from cap.paths import get_paths
from cap.provenance import validate_frozen_inputs, write_run_provenance, sha256_file

paths = get_paths()

raw_dir = paths.raw_counts_dir
features_file = paths.features_file
policy_file = paths.frozen_dir / "gene_policy.json"
cohort_file = paths.root / "config" / COHORT_RELEASE_NAME
preprocessing_notebook = paths.root / "notebooks" / "001_Preprocessing.ipynb"

cohort_stage_dir = paths.stage_dir("00_cohort")
cohort_samples_file = cohort_stage_dir / "tables" / "cohort_samples.csv"
cohort_checksums_file = cohort_stage_dir / "input_checksums.json"
cohort_provenance_file = cohort_stage_dir / "run_provenance.json"

driver_notebook = paths.root / "notebooks" / "001_Papermill.ipynb"
preprocessing_notebook = paths.root / "notebooks" / "001_Preprocessing.ipynb"

for path, description in (
        (cohort_samples_file, "00_cohort sample sheet"),
        (cohort_checksums_file, "00_cohort input checksums"),
        (cohort_provenance_file, "00_cohort provenance"),
        (driver_notebook, "001 Papermill driver"),
        (preprocessing_notebook, "001 preprocessing notebook"),
):
    paths.require_file(path, description)

cohort_provenance = json.loads(
    cohort_provenance_file.read_text(encoding="utf-8")
)
if cohort_provenance.get("run_id") != paths.run_id:
    raise ValueError(
        "The 00_cohort run ID does not match the current CAP_RUN_ID."
    )
if cohort_provenance.get("git_dirty") is not False:
    raise ValueError("The 00_cohort provenance records a dirty Git state.")
if not cohort_provenance.get("provenance_completeness", {}).get("complete"):
    raise ValueError("The 00_cohort provenance is incomplete.")
if sha256_file(cohort_checksums_file) != cohort_provenance.get(
        "input_checksums_file_sha256"
):
    raise ValueError("The 00_cohort input-checksum file does not match its provenance.")

paths.require_dir(raw_dir, "GSE161529 filtered count directory")
paths.require_file(features_file, "shared features.tsv")
paths.require_file(policy_file, "frozen gene policy")
paths.require_file(cohort_file, "frozen cohort release")
paths.require_file(preprocessing_notebook, "001 preprocessing notebook")
validate_frozen_inputs(paths)

def display_path(path):
    """Return a project-relative display path without local absolute paths."""
    resolved = Path(path).resolve()
    for root, label in (
        (paths.data_root, "<CAP_DATA_ROOT>"),
        (paths.root, "<REPOSITORY>"),
    ):
        try:
            relative = resolved.relative_to(Path(root).resolve())
            return f"{label}/{relative.as_posix()}"
        except ValueError:
            continue
    return f"<external>/{resolved.name}"


## Select included samples

In [3]:
cohort = pd.read_csv(cohort_file)

if "sample_id" not in cohort.columns:
    raise KeyError(
        f"Frozen cohort must contain a 'sample_id' column; found: {list(cohort.columns)}"
    )

cohort["sample_id"] = cohort["sample_id"].astype("string").str.strip()
if cohort["sample_id"].isna().any() or cohort["sample_id"].eq("").any():
    raise ValueError("Frozen cohort contains a missing or empty sample_id.")
if cohort["sample_id"].duplicated().any():
    duplicates = cohort.loc[cohort["sample_id"].duplicated(), "sample_id"].tolist()
    raise ValueError(f"Duplicate sample_id values in frozen cohort: {duplicates}")

if "include" in cohort.columns:
    include_map = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }
    include = (
        cohort["include"].astype("string").str.strip().str.lower().map(include_map)
    )
    if include.isna().any():
        bad_values = cohort.loc[include.isna(), "include"].drop_duplicates().tolist()
        raise ValueError(f"Unrecognised values in cohort 'include' column: {bad_values}")
    eligible_ids = cohort.loc[include, "sample_id"].astype(str).tolist()
else:
    # A release containing only accepted samples does not need an include column.
    eligible_ids = cohort["sample_id"].astype(str).tolist()

stage_cohort = pd.read_csv(cohort_samples_file)

required_columns = {"sample_id", "include"}
missing_columns = required_columns.difference(stage_cohort.columns)
if missing_columns:
    raise KeyError(
        f"00_cohort sample sheet is missing columns: {sorted(missing_columns)}"
    )

stage_cohort["sample_id"] = (
    stage_cohort["sample_id"].astype("string").str.strip()
)
if stage_cohort["sample_id"].isna().any() or stage_cohort["sample_id"].eq("").any():
    raise ValueError("00_cohort sample sheet contains a missing or empty sample_id.")

stage_include = (
    stage_cohort["include"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False})
)
if stage_include.isna().any():
    raise ValueError("00_cohort sample sheet has unrecognised include values.")

stage_ids = set(stage_cohort.loc[stage_include, "sample_id"].astype(str))
release_ids = set(eligible_ids)

if stage_ids != release_ids:
    raise ValueError(
        "The included sample IDs in the frozen cohort release do not match "
        "the 00_cohort sample sheet. "
        f"Only in 00_cohort: {sorted(stage_ids - release_ids)}; "
        f"only in frozen release: {sorted(release_ids - stage_ids)}"
    )

if RUN_DEMO:
    missing_demo_ids = sorted(set(DEMO_IDS).difference(eligible_ids))
    if missing_demo_ids:
        raise ValueError(
            f"Demo sample(s) are not included in the frozen cohort: {missing_demo_ids}"
        )
    sample_ids = list(DEMO_IDS)
else:
    sample_ids = eligible_ids

if not sample_ids:
    raise ValueError("No samples were selected for preprocessing.")

print(f"Frozen cohort: {display_path(cohort_file)}")
print(f"Selected samples: {len(sample_ids)}" + (" (demo)" if RUN_DEMO else ""))


Frozen cohort: <REPOSITORY>/config/cohort_release_2026-11-13.csv
Selected samples: 2 (demo)


## Prepare the preprocessing stage

In [4]:
import os
import shutil
import subprocess

print("paths.root:", paths.root)
print("git executable:", shutil.which("git"))
print("Git environment:", {
    key: os.environ.get(key)
    for key in ("GIT_DIR", "GIT_WORK_TREE", "GIT_COMMON_DIR", "GIT_INDEX_FILE")
})

probe = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=paths.root,
    capture_output=True,
    text=True,
)
print("Git probe return code:", probe.returncode)
print("Git probe stdout:", probe.stdout)
print("Git probe stderr:", probe.stderr)

paths.root: D:\IdeaProjects\CAP_Submission\CAP_Analysis
git executable: C:\Program Files\Git\cmd\git.EXE
Git environment: {'GIT_DIR': None, 'GIT_WORK_TREE': None, 'GIT_COMMON_DIR': None, 'GIT_INDEX_FILE': None}
Git probe return code: 0
Git probe stdout: f4917e033b7f9b448a6257c322d2c32d30521e44

Git probe stderr: 


In [5]:
input_files = {
    COHORT_RELEASE_NAME: cohort_file,
    "00_cohort/cohort_samples.csv": cohort_samples_file,
    "00_cohort/input_checksums.json": cohort_checksums_file,
    "00_cohort/run_provenance.json": cohort_provenance_file,
    "features.tsv": features_file,
    "gene_policy.json": policy_file,
    "001_Preprocessing.ipynb": preprocessing_notebook,
}
input_checksums = {
    name: sha256_file(path)
    for name, path in sorted(input_files.items())
}

# Refuse to overwrite an existing stage; use a fresh CAP_RUN_ID instead.
stage_dir = paths.prepare_stage("01_preprocessing")

executed_dir = stage_dir / "executed_notebooks"
log_dir = stage_dir / "logs"
tables_dir = stage_dir / "tables"
for directory in (executed_dir, log_dir, tables_dir):
    directory.mkdir(parents=True, exist_ok=True)

checksums_file = stage_dir / "input_checksums.json"
checksums_file.write_text(
    json.dumps(input_checksums, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

write_run_provenance(
    paths,
    stage_dir,
    notebook_path=driver_notebook,
    input_checksums=input_checksums,
)

WindowsPath('D:/CAP_Data/runs/cap_preprocessing_20261120_210737/01_preprocessing/run_provenance.json')

## Execute notebook 001 for each selected sample

In [6]:
completed_ids = []
failed_ids = []

for sample_id in sample_ids:
    executed_file = executed_dir / f"001_{sample_id}.ipynb"
    partial_file = executed_dir / f"001_{sample_id}.partial.ipynb"
    log_file = log_dir / f"{sample_id}.log"

    print(f"Run {sample_id}")
    try:
        with log_file.open("w", encoding="utf-8") as log_handle:
            pm.execute_notebook(
                input_path=str(preprocessing_notebook),
                output_path=str(partial_file),
                parameters={"sample_id": sample_id},
                kernel_name=KERNEL_NAME,
                progress_bar=False,
                log_output=True,
                stdout_file=log_handle,
                stderr_file=log_handle,
                cwd=str(paths.root),
            )

        partial_file.replace(executed_file)
        completed_ids.append(sample_id)

    except Exception as exc:
        failed_ids.append(sample_id)
        print(
            f"FAILED {sample_id}: {type(exc).__name__}: {exc} "
            f"(log: {display_path(log_file)})"
        )

print(
    f"\nCompleted: {len(completed_ids)}  Failed: {len(failed_ids)}  "
    f"(of {len(sample_ids)} selected)"
)
if failed_ids:
    raise RuntimeError(
        f"{len(failed_ids)} sample(s) failed: {', '.join(failed_ids)}. "
        "See the per-sample logs and partial notebooks."
    )

Run GSM4909292_HER2-MH0069
FAILED GSM4909292_HER2-MH0069: PapermillExecutionError: 
---------------------------------------------------------------------------
Exception encountered at "In [10]":
---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
Cell In[10], line 86
     82     json.dumps(input_checksums, indent=2, sort_keys=True) + "\n",
     83     encoding="utf-8",
     84 )
     85 
---> 86 write_run_provenance(
     87     paths,
     88     sample_table_dir,
     89     notebook_path=notebook_path,

File D:\IdeaProjects\CAP_Submission\CAP_Analysis\src\cap\provenance.py:144, in write_run_provenance(paths, stage_dir, notebook_path, input_checksums)
    137 git_result = subprocess.run(
    138     ["git", "rev-parse", "HEAD"],
    139     cwd=root,
    140     text=True,
    141     capture_output=True,
    142 )
    143 if git_result.returncode:
--> 144     raise RuntimeError(
    

RuntimeError: 2 sample(s) failed: GSM4909292_HER2-MH0069, GSM4909272_N-MH0021-Total. See the per-sample logs and partial notebooks.

## Combine per-sample QC tables

In [ ]:
qc_files = {
    sample_id: (
        tables_dir / "by_sample" / sample_id / "qc_summary.csv"
    )
    for sample_id in sample_ids
}
missing_qc = [sample_id for sample_id, path in qc_files.items() if not path.is_file()]
if missing_qc:
    missing_paths = [display_path(qc_files[sample_id]) for sample_id in missing_qc]
    raise FileNotFoundError(
        f"QC table(s) missing for {len(missing_qc)} sample(s): {missing_paths}"
    )

qc_summary = pd.concat(
    [pd.read_csv(qc_files[sample_id]) for sample_id in sample_ids],
    ignore_index=True,
)

if len(qc_summary) != len(sample_ids):
    raise ValueError(
        f"Expected one QC row per selected sample ({len(sample_ids)}), "
        f"found {len(qc_summary)} rows."
    )

sort_columns = [column for column in ("condition", "sample") if column in qc_summary]
if sort_columns:
    qc_summary = qc_summary.sort_values(sort_columns).reset_index(drop=True)

sample_column = next(
    (column for column in ("sample", "sample_id") if column in qc_summary),
    None,
)
if sample_column and qc_summary[sample_column].duplicated().any():
    raise ValueError(f"Duplicate sample rows in QC summary column {sample_column!r}.")

suffix = "_demo" if RUN_DEMO else ""
qc_summary_file = tables_dir / f"qc_summary{suffix}.csv"
qc_summary.to_csv(qc_summary_file, index=False, lineterminator="\n")

print(f"Written: {display_path(qc_summary_file)} ({len(qc_summary)} samples)")
